In [27]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from scorecard.data import load_german_credit
from scorecard.binning import bin_categorical, bin_continuous
from scorecard.woe import compute_woe_iv, apply_woe

pd.set_option("display.max_columns", None)

SEED = 42

In [28]:
df = load_german_credit("../data/raw/german_credit/german.data")
df.head()

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_rate,personal_status_sex,other_parties,residence_since,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,target
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,4,A121,67,A143,A152,2,A173,1,A192,A201,0
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,2,A121,22,A143,A152,1,A173,1,A191,A201,1
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,3,A121,49,A143,A152,1,A172,2,A191,A201,0
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,4,A122,45,A143,A153,1,A173,2,A191,A201,0
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,4,A124,53,A143,A153,2,A173,2,A191,A201,1


In [29]:
categorical_cols = df.select_dtypes(include=["str"]).columns.tolist()
continuous_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
continuous_cols.remove("target")

In [30]:
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y,random_state=SEED)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(y_train.mean(), y_test.mean())

Train shape: (800, 20), Test shape: (200, 20)
0.3 0.3


In [31]:
iv_results = {}
woe_tables = {}
train_binned_cols = {}
test_binned_cols = {}

for col in categorical_cols:
    train_b, test_b, rare = bin_categorical(X_train[col], X_test[col])
    summary, iv = compute_woe_iv(train_b, y_train)
    train_binned_cols[col] = train_b
    test_binned_cols[col] = test_b
    woe_tables[col] = summary
    iv_results[col] = iv

for col in continuous_cols:
    train_b, test_b, edges = bin_continuous(X_train[col], X_test[col])
    summary, iv = compute_woe_iv(train_b, y_train)
    train_binned_cols[col] = train_b
    test_binned_cols[col] = test_b
    woe_tables[col] = summary
    iv_results[col] = iv

iv_ranking = pd.Series(iv_results).sort_values(ascending=False)
iv_ranking

checking_status        0.611906
credit_history         0.264045
savings_status         0.215376
duration               0.214537
purpose                0.147901
property_magnitude     0.145841
employment             0.124940
other_payment_plans    0.091335
housing                0.088983
credit_amount          0.065846
personal_status_sex    0.058019
foreign_worker         0.046856
age                    0.043165
installment_rate       0.031031
other_parties          0.025280
job                    0.012364
own_telephone          0.008201
existing_credits       0.001248
residence_since        0.000111
num_dependents         0.000000
dtype: float64

In [32]:
keep_features = iv_ranking[iv_ranking > 0.02].index.tolist() # type: ignore
print(f"Keeping {len(keep_features)} of {len(iv_ranking)} features with IV > 0.02")
print(keep_features)

Keeping 15 of 20 features with IV > 0.02
['checking_status', 'credit_history', 'savings_status', 'duration', 'purpose', 'property_magnitude', 'employment', 'other_payment_plans', 'housing', 'credit_amount', 'personal_status_sex', 'foreign_worker', 'age', 'installment_rate', 'other_parties']


In [33]:
from scorecard.woe import apply_woe

X_train_woe_full = pd.DataFrame(index=X_train.index)
X_test_woe_full = pd.DataFrame(index=X_test.index)

for col in keep_features:
    X_train_woe_full[col] = apply_woe(train_binned_cols[col], woe_tables[col])
    X_test_woe_full[col] = apply_woe(test_binned_cols[col], woe_tables[col])

In [34]:
for col in keep_features:
    n_nan = X_test_woe_full[col].isna().sum()
    if n_nan > 0:
        print(f"{col}: {n_nan} NaNs")

In [ ]:
corr_matrix = X_train_woe_full[keep_features].corr()
print(corr_matrix)

,checking_status,credit_history,savings_status,duration,purpose,property_magnitude,employment,other_payment_plans,housing,credit_amount,personal_status_sex,foreign_worker,age,installment_rate,other_parties
checking_status,1.000000,0.173215,0.243923,0.077596,0.125680,0.086878,0.117664,0.064663,0.113896,0.066527,0.047391,-0.020408,0.107638,0.007510,-0.009548
credit_history,0.173215,1.000000,0.056141,0.071947,0.039471,0.071600,0.127465,0.200881,0.110729,0.028106,0.058741,0.046731,0.119016,-0.067014,-0.018395
savings_status,0.243923,0.056141,1.000000,-0.008200,0.062865,-0.008239,0.105650,0.001572,-0.002583,-0.036378,0.007946,0.018368,0.083684,-0.059642,-0.007757
duration,0.077596,0.071947,-0.008200,1.000000,-0.065815,0.238193,-0.029520,0.080134,0.041670,0.390230,-0.064682,0.146888,-0.008148,0.076490,0.052280
purpose,0.125680,0.039471,0.062865,-0.065815,1.000000,0.058276,0.071025,0.042900,0.036528,-0.040761,0.080969,-0.072869,-0.034719,-0.004017,0.030600
property_magnitude,0.086878,0.071600,-0.008239,0.238193,0.058276,1.000000,-0.007373,0.098973,0.404927,0.255601,-0.100225,0.099968,-0.110155,0.042729,0.110035
employment,0.117664,0.127465,0.105650,-0.029520,0.071025,-0.007373,1.000000,-0.000335,0.046086,0.004130,0.205472,-0.003398,0.122358,-0.060250,0.072095
other_payment_plans,0.064663,0.200881,0.001572,0.080134,0.042900,0.098973,-0.000335,1.000000,0.010663,0.049676,-0.044323,0.020712,-0.063230,0.004073,-0.046057
housing,0.113896,0.110729,-0.002583,0.041670,0.036528,0.404927,0.046086,0.010663,1.000000,0.096664,0.065816,0.016223,0.114934,-0.045720,0.065970
credit_amount,0.066527,0.028106,-0.036378,0.390230,-0.040761,0.255601,0.004130,0.049676,0.096664,1.000000,-0.065731,0.078149,-0.027493,-0.147978,0.030340


In [36]:
model_full = LogisticRegression(random_state=SEED)
model_full.fit(X_train_woe_full[keep_features], y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following a

In [37]:
p_test_full = model_full.predict_proba(X_test_woe_full[keep_features])[:, 1]
auc_full = roc_auc_score(y_test, p_test_full)
gini_full = 2 * auc_full - 1
print(f"AUC: {auc_full:.4f}, Gini: {gini_full:.4f}")

AUC: 0.8019, Gini: 0.6038


In [38]:
pd.Series(model_full.coef_[0], index=keep_features).sort_values()

installment_rate      -1.326312
personal_status_sex   -0.926067
purpose               -0.898812
duration              -0.836322
savings_status        -0.832152
checking_status       -0.770131
foreign_worker        -0.766607
other_payment_plans   -0.752110
credit_history        -0.671955
other_parties         -0.655694
employment            -0.608161
age                   -0.584009
credit_amount         -0.479629
property_magnitude    -0.395596
housing               -0.349129
dtype: float64